In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.models import ResNet50_Weights

import random
import copy

In [ ]:
data_dir = Path('./data/persons')
batch_size = 4
num_workers = 2
num_epochs = 10
learning_rate_head = 0.001
learning_rate_finetune = 0.00001
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

def set_seed(seed:int=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def evaluate(model, loader, criterion, device):
    # TODO

def train_one_epoch(model, loader, criterion, optimizer, device):
    # TODO

def is_valid_image(path: str) -> bool:
    return Path(path).suffix.lower() == '.jpg'

def build_balanced_split_indices(image_folder_dataset, seed=42):
    rng = random.Random(seed)
    class_to_indices = {}
    for idx, (_, label) in enumerate(image_folder_dataset.samples):
        class_to_indices.setdefault(label, []).append(idx)
    train_indices = []
    val_indices = []
    for cls_ids, idxs in class_to_indices.items():
        rng.shuffle(idxs)
        if len(idxs) < 2:
            raise ValueError(f"Not enough samples for class {cls_ids} to create a train/val split.")
        val_indices.extend(idxs[0:3])
        train_indices.extend(idxs[3:])
    return train_indices, val_indices



In [ ]:
set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")    

if not data_dir.exists():
    raise FileNotFoundError(f"Data directory {data_dir} does not exist. Please ensure the dataset is available.")

transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

val_trfms = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

full_for_split = datasets.ImageFolder(root=data_dir, transform=transform, is_valid_file=is_valid_image)
class_names = full_for_split.classes
num_classes = len(class_names)
print(f"Classes found: {class_names}")
print(f"Number of classes: {num_classes}")
print(f"Total images found: {len(full_for_split)}")

train_indices, val_indices = build_balanced_split_indices(full_for_split, seed=42)
train_base = datasets.ImageFolder(root=data_dir, transform=transform, is_valid_file=is_valid_image)
val_base = datasets.ImageFolder(root=data_dir, transform=val_trfms, is_valid_file=is_valid_image)

train_ds = Subset(train_base, train_indices)
val_ds = Subset(val_base, val_indices)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)

In [ ]:
# ResNet-50 model with pretrained with ImageNet weights
# TODO

model.load_state_dict(best_model_wts)
torch.save(model.state_dict(), 'best_model_head.pth')



In [ ]:
# Open one image randomly and test
import matplotlib.pyplot as plt
from PIL import Image

def predict_image(model, image_path, transform, device):
    model.eval()
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(image_tensor)
        probs = torch.softmax(logits, dim=1)
        pred_idx = probs.argmax().item()
        pred_class = class_names[pred_idx]
        pred_prob = probs[0, pred_idx].item()
    return pred_class, pred_prob


random_image_path = random.choice(list(data_dir.glob('**/*.jpg')))
pred_class, pred_prob = predict_image(model, random_image_path, val_trfms, device)
print(f"Predicted class: {pred_class}, Probability: {pred_prob:.4f}")
image = Image.open(random_image_path).convert('RGB')
plt.imshow(image)
plt.title(f"Predicted: {pred_class} ({pred_prob:.2f})")
plt.axis('off')
plt.show()

In [ ]:
# Fine-tune the entire model
# TODO



In [ ]:
# Open one image randomly and test
import matplotlib.pyplot as plt
from PIL import Image

def predict_image(model, image_path, transform, device):
    model.eval()
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(image_tensor)
        probs = torch.softmax(logits, dim=1)
        pred_idx = probs.argmax().item()
        pred_class = class_names[pred_idx]
        pred_prob = probs[0, pred_idx].item()
    return pred_class, pred_prob


random_image_path = random.choice(list(data_dir.glob('**/*.jpg')))
pred_class, pred_prob = predict_image(model, random_image_path, val_trfms, device)
print(f"Predicted class: {pred_class}, Probability: {pred_prob:.4f}")
image = Image.open(random_image_path).convert('RGB')
plt.imshow(image)
plt.title(f"Predicted: {pred_class} ({pred_prob:.2f})")
plt.axis('off')
plt.show()